In [ ]:
!pip install pydantic

In [20]:
import pandas as pd
import json
import time
from tqdm import tqdm
from pydantic import BaseModel, Field
from openai import OpenAI
import os

# ==========================================
# CONFIGURAÇÃO
# ==========================================
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Carregando os dados
df = pd.read_json("./videos.jsonl", lines=True)
print(len(df))
df['texto_leve'] = df['title'].fillna('') + '\n' + df['description'].fillna('')

# ==========================================
# PASSO 1: AMOSTRAGEM PARA DESCOBERTA
# ==========================================
print("Gerando amostra para criação da taxonomia...")
df_amostra = df.sample(n=150, random_state=42)

# Truncando em 1.000 caracteres para a amostra caber confortavelmente no prompt
textos_amostra = "\n\n---\n\n".join(
    [f"Documento {i+1}:\n{texto[:1000]}" for i, texto in enumerate(df_amostra['texto_leve'])]
)

prompt_geracao = f"""
Você é um pesquisador focado em análise de discurso de saúde pública no YouTube.
Abaixo, forneço uma amostra de textos (Título + Início da Descrição) de vídeos focados em medicamentos para emagrecimento.

Seu objetivo:
Criar uma taxonomia de 5 a 7 tópicos mutuamente exclusivos que representem a INTENÇÃO PRINCIPAL ou TEMA CENTRAL do vídeo. 
Dê um peso especial ao Título, pois ele geralmente revela a intenção do criador do conteúdo.

Regras:
1. Ignore ruídos (pedidos de like, links de afiliados, CRMs médicos, contatos).
2. Ignore o fato de que "falam de Ozempic/emagrecer" (isso é o viés comum de todos os vídeos). Foque no contexto (ex: é relato de efeito colateral? Propaganda de consulta? Dúvida técnica?).
3. Liste os tópicos com um Nome Curto e uma breve descrição do que entra nele. Tente evitar sobreposição entre os tópicos.

Amostra de documentos:
{textos_amostra}
"""



3027
Gerando amostra para criação da taxonomia...


In [21]:
print("Solicitando taxonomia ao modelo...")
resposta_taxonomia = client.chat.completions.create(
    model="gpt-4o", # O modelo maior é melhor para a fase qualitativa inicial
    messages=[{"role": "user", "content": prompt_geracao}]
)
print(resposta_taxonomia.choices[0].message.content)



Solicitando taxonomia ao modelo...
Baseado na análise dos documentos fornecidos, aqui está uma taxonomia de tópicos mutuamente exclusivos que capturam a intenção ou o tema central dos vídeos mencionados:

1. **Comparação e Escolha de Medicamentos**
   - Estes vídeos comparam diferentes medicamentos para emagrecimento, como Ozempic, Wegovy e Mounjaro, discutindo qual pode ser mais eficaz ou adequado para determinados usuários.
   - Exemplos: Documentos 2, 19, 32, 49.

2. **Aplicação e Uso Correto**
   - Foco em orientar os usuários sobre como aplicar corretamente os medicamentos, a dosagem recomendada e cuidados durante o uso.
   - Exemplos: Documentos 3, 13, 50, 97.

3. **Efeitos Colaterais e Riscos**
   - Discutem os efeitos adversos dos medicamentos, como náuseas ou flacidez, e oferecem dicas de como minimizá-los.
   - Exemplos: Documentos 10, 18, 68, 128.

4. **Questões de Saúde Mental e Comportamento**
   - Abordam a relação entre o uso de emagrecimentos e aspectos psicológicos ou 

In [25]:
# ==========================================
# PASSO 2: DEFINIÇÃO DO CODEBOOK E SCHEMA PYDANTIC
# ==========================================
# Substitua esta lista pelos tópicos que o modelo gerar no Passo 1 após a sua validação humana
CATEGORIAS_PERMITIDAS = [
    "Comparação e Escolha de Medicamentos",
    "Aplicação e Uso Correto",
    "Efeitos Colaterais e Riscos",
    "Questões de Saúde Mental e Comportamento",
    "Eficácia, Resultados e Relatos Pessoais",
    "Considerações Médicas e Recomendação de Profissionais",
    "Aspectos Econômicos e Acessibilidade"
]


In [28]:
class ClassificacaoVideo(BaseModel):
    topico: str = Field(
        description="A categoria exata à qual o vídeo pertence.", 
        enum=CATEGORIAS_PERMITIDAS
    )

# ==========================================
# PASSO 3: CLASSIFICAÇÃO EM LOTE (ATRIBUIÇÃO)
# ==========================================
def classificar_video(texto: str) -> dict:
    prompt_sistema = f"""
    Você é um classificador estruturado. Sua tarefa é analisar o texto (Título + Descrição) de um vídeo do YouTube e categorizá-lo.
    O título costuma ditar a intenção real. Ignore pedidos de like, inscrições de canal, links e avisos legais.
    """
    
    try:
        response = client.beta.chat.completions.parse(
            model=MODELO,
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": f"Classifique este texto:\n{texto}"}
            ],
            response_format=ClassificacaoVideo,
            temperature=0.0 # Temperatura zero para taxonomia estrita
        )
        return response.choices[0].message.parsed.model_dump()
        
    except Exception as e:
        # Removida a justificativa daqui também para acompanhar seu modelo atualizado
        return {"topico": "Erro_API"}

# ==========================================
# EXECUÇÃO DO PIPELINE NAS 3.000 LINHAS
# ==========================================
print("\nIniciando classificação estruturada...")

ARQUIVO_CHECKPOINT = "resultados_parciais.json"

# 1. Carrega o progresso anterior, se o arquivo existir
if os.path.exists(ARQUIVO_CHECKPOINT):
    with open(ARQUIVO_CHECKPOINT, "r", encoding="utf-8") as f:
        resultados = json.load(f)
    # Cria um set (conjunto) para busca ultra-rápida (O(1)) dos IDs já processados
    ids_processados = set(res["video_id"] for res in resultados)
    print(f"Progresso encontrado! Retomando a partir de {len(ids_processados)} vídeos já processados.")
else:
    resultados = []
    ids_processados = set()

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classificando vídeos"):
    video_id = row.get('video_id', idx)
    
    # 2. Pula a iteração se o vídeo já foi classificado em uma execução anterior
    if video_id in ids_processados:
        continue
    
    texto_truncado = str(row['texto_leve'])[:2500] 
    
    if not texto_truncado.strip():
        resultados.append({
            "video_id": video_id,
            "topico_llm": "Outros_Irrelevante"
        })
    else:
        classificacao = classificar_video(texto_truncado)
        resultados.append({
            "video_id": video_id,
            "topico_llm": classificacao["topico"]
        })
    
    # 3. Adiciona o ID na lista de segurança e salva o arquivo de checkpoint
    ids_processados.add(video_id)
    
    with open(ARQUIVO_CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)
    
    time.sleep(0.05) 

# ==========================================
# CONSOLIDAÇÃO FINAL
# ==========================================
df_resultados = pd.DataFrame(resultados)
df_final = pd.merge(df, df_resultados, on="video_id", how="left")

# Salvando usando formatação e encoding limpos
df_final.to_csv("classificacao_topicgpt_openai_final.csv", index=False, encoding='utf-8-sig')

print("\n[OK] Pipeline concluído! Resultados salvos.")
print("\nDistribuição dos Tópicos Encontrados:")
print(df_final['topico_llm'].value_counts())


Iniciando classificação estruturada...
Progresso encontrado! Retomando a partir de 3 vídeos já processados.


Classificando vídeos: 100%|████████████████████████████████████████████████████████| 3027/3027 [55:38<00:00,  1.10s/it]



[OK] Pipeline concluído! Resultados salvos.

Distribuição dos Tópicos Encontrados:
topico_llm
Eficácia, Resultados e Relatos Pessoais                  1238
Efeitos Colaterais e Riscos                               692
Aplicação e Uso Correto                                   373
Comparação e Escolha de Medicamentos                      315
Considerações Médicas e Recomendação de Profissionais     192
Aspectos Econômicos e Acessibilidade                      110
Questões de Saúde Mental e Comportamento                  107
Name: count, dtype: int64


In [32]:
df_final.to_json('./videos_classificados.json', lines=True, orient='records', force_ascii=False)